## Introduccion

La Transformada Discreta de Fourier (DFT) permite representar una señal discreta en el dominio de la frecuencia. Para una señal de \(N\) muestras obtenidas con una frecuencia de muestreo \(f_s\), la separación entre los distintos puntos donde la DFT puede representarlo se denomina resolucion espectral \(\Delta f\) y se calcula como:

$$
\Delta f=\frac{f_s}{N}
$$

Cada punto de frecuencia de la DFT se identifica mediante un índice entero \(k\), denominado **bin frecuencial**. El índice \(k\) indica la posición del punto dentro del espectro, y la frecuencia asociada a dicho bin se obtiene mediante:

$$
f_k=k\Delta f
$$

Por lo tanto, \(k\) representa el índice que permite identificar cada bin de la DFT.

 Se utiliza además \(k_0\), que representa el índice de frecuencia elegido para definir la frecuencia de la señal senoidal. La frecuencia de la senoide, \(f_0\), se obtiene mediante:

$$
f_0=k_0\Delta f
$$

De esta manera, \(k_0\) permite controlar si la frecuencia de la senoide coincide o no con uno de los bins de la DFT.

Cuando \(f_0\) coincide exactamente con un bin de la DFT, la potencia espectral de la senoide se concentra en dicho bin. En cambio, cuando \(f_0\) se encuentra entre dos bins, la potencia se distribuye entre varios bins. 

Para analizar la distribución de potencia en frecuencia, se calcula la DFT de cada señal. Los coeficientes de la DFT son, en general, números complejos. La potencia asociada a cada componente frecuencial se obtiene mediante el módulo al cuadrado de dichos coeficientes:

$$
PDS[k]=|X[k]|^2
$$

La **identidad de Parseval** establece que la energía o potencia de una señal puede calcularse de forma equivalente tanto en el dominio temporal como en el dominio frecuencial. 

$$
\frac{1}{N}\sum_{n=0}^{N-1}|x[n]|^2
=
\sum_{k=0}^{N-1}|X[k]|^2
$$

Por lo tanto, Parseval permite verificar que la potencia total obtenida a partir de la PDS coincide con la potencia de la señal original. En este experimento se utilizan senoidales de potencia unitaria, por lo que la suma de las potencias de los distintos componentes frecuenciales debe ser igual a uno.


In [ ]:
# -*- coding: utf-8 -*-

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

#%% Parámetros del sistema
N  = 1000
fs = 1000

delta_f = fs / N 
 
vmax = np.sqrt(2)

dc = 0
ph = np.pi/2


#%%

# ff frecuencia de la senoide, nn cantidad de muestras
def mi_funcion_sen(vmax, dc, ff, ph, nn, fs):
    tt = np.arange(0, nn / fs, 1 / fs)
    xx = dc + vmax * np.sin(2 * np.pi * tt * ff + ph)
    return (tt, xx)

#%% Primera

k_o = N/4 #donde pongo la senoidal 
f_o = k_o * delta_f #Frecuencia de la senoidal 

tt, xx1 = mi_funcion_sen(vmax=vmax, dc=dc, ff=f_o, ph=ph, nn=N, fs=fs)

#%% Segunda

k_o = N/4 + 0.25
f_o = k_o * delta_f

tt, xx2 = mi_funcion_sen(vmax=vmax, dc=dc, ff=f_o, ph=ph, nn=N, fs=fs)

#%% Tercera

k_o = N/4 + 0.5
f_o = k_o * delta_f

tt, xx3 = mi_funcion_sen(vmax=vmax, dc=dc, ff=f_o, ph=ph, nn=N, fs=fs)

#%% fft

XX1 = 1/N * np.fft.fft(xx1)
XX2 = 1/N * np.fft.fft(xx2)
XX3 = 1/N * np.fft.fft(xx3)

frecs = np.arange(N) * delta_f

#%%

PDS1 = np.abs(XX1)**2
PDS2 = np.abs(XX2)**2
PDS3 = np.abs(XX3)**2


#%% Gráfico PDS

plt.figure()

plt.plot(frecs, PDS1, label='f0 = 250 Hz')
plt.plot(frecs, PDS2, label='f0 = 250.25 Hz')
plt.plot(frecs, PDS3, label='f0 = 250.5 Hz')

plt.xlabel('Frecuencia [Hz]')
plt.ylabel('Potencia')
plt.title('Desparramo espectral')
plt.legend()
plt.grid()

plt.xlim(240, 260)

plt.show()

### Análisis

Como se puede observar en el gráfico, para k0=250 se obtiene una frecuencia f0=250 Hz, que coincide exactamente con uno de los bins de la DFT. Por esto, la potencia espectral se concentra en 250 Hz.

Sin embargo, para k0=250.25 y k0=250.5, las frecuencias obtenidas son 250.25 Hz y 250.5 Hz, respectivamente, y no coinciden con un bin de la DFT. En estos casos, la potencia se distribuye entre varios bins, produciendo desparramo espectral.

A medida que la frecuencia de la senoide se encuentra más desintonizada respecto de los bins de la DFT, se modifica la distribución de la potencia en el espectro.


In [ ]:
P1 = np.sum(PDS1)
P2 = np.sum(PDS2)
P3 = np.sum(PDS3)

print(P1)
print(P2)
print(P3)

### Item B

Al sumar los valores de cada PDS se obtiene una potencia aproximadamente unitaria para las tres señales: \(1.000\), \(1.001\) y \(1.000\), verificándose así la potencia unitaria planteada. 

La diferencia observada entre los espectros se debe a la relación entre la frecuencia de la senoide y la separación entre los bins de la DFT. Para f0=250 Hz, la frecuencia coincide exactamente con un bin, por lo que la potencia se concentra principalmente en él. En cambio, para las demás, la frecuencia se encuentra entre dos bins, provocando que la potencia se distribuya entre varios bins y produciendo el fenómeno de desparramo espectral.

Por este motivo, una pequeña diferencia en la frecuencia de la senoide puede producir una diferencia importante en el espectro observable, aunque la potencia total de las señales permanezca aproximadamente constante.


In [ ]:
#%% Zero padding

N_zp = 10 * N
delta_f_zp = fs / N_zp

xx1_zp = np.concatenate((xx1, np.zeros(9*N)))
xx2_zp = np.concatenate((xx2, np.zeros(9*N)))
xx3_zp = np.concatenate((xx3, np.zeros(9*N)))

#%% DFT

XX1_zp = 1/N * np.fft.fft(xx1_zp)
XX2_zp = 1/N * np.fft.fft(xx2_zp)
XX3_zp = 1/N * np.fft.fft(xx3_zp)

frecs_zp = np.arange(N_zp) * delta_f_zp

#%% PDS

PDS1_zp = np.abs(XX1_zp)**2
PDS2_zp = np.abs(XX2_zp)**2
PDS3_zp = np.abs(XX3_zp)**2

#%% Graf

plt.figure()

plt.plot(frecs_zp, PDS1_zp, label='f0 = 250 Hz')
plt.plot(frecs_zp, PDS2_zp, label='f0 = 250.25 Hz')
plt.plot(frecs_zp, PDS3_zp, label='f0 = 250.5 Hz')

plt.xlabel('Frecuencia [Hz]')
plt.ylabel('Potencia')
plt.title('Desparramo espectral con Zero Padding')
plt.legend()
plt.grid()

plt.xlim(240, 260)

plt.show()

Al aumentar el número de muestras utilizadas en la DFT mediante zero padding, aumenta la resolución espectral, ya que:

$$
\Delta f=\frac{f_s}{N}
$$

Al agregar \(9N\) ceros, el número total de puntos pasa de \(N=1000\) a \(N=10000\), por lo que la separación entre los puntos de frecuencia disminuye de 1 Hz a 0.1 Hz

Esto permite representar el espectro con mayor cantidad de puntos y observar con mayor detalle la distribución de la potencia. Sin embargo, el zero padding no elimina el desparramo espectral ni agrega información nueva a la señal. 

### Conclusión

A partir del experimento se pudo observar el efecto del desparramo espectral producido cuando la frecuencia de una señal senoidal no coincide con un bin de la DFT. Cuando la frecuencia coincide con un bin, la potencia se concentra principalmente en dicho punto, mientras que una pequeña desintonía provoca que la potencia se distribuya entre varios bins.

Además, se verificó que la potencia total de las señales permanece aproximadamente unitaria, a pesar de las diferencias en la distribución de la potencia espectral. Finalmente, mediante zero padding se aumentó la cantidad de puntos utilizados para representar el espectro, reduciendo \(\Delta f\) de 1 Hz a 0.1 Hz. Esto permite visualizar el espectro con mayor detalle, sin agregar información nueva a la señal.
